# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/darksider747/flyrank-1st/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Archetype -> Action mapping (from Week 4's reason codes):

- ctr_and_stale (low CTR for position + not updated in 91+ days)
  -> high_priority_review
- ctr_below_tier_average (low CTR for position, otherwise fine)
  -> review_title_meta
- stale_content (not updated in 91+ days, CTR otherwise fine)
  -> review_content_refresh
- no_flag (neither issue present) -> monitor

Each page is first sorted into one of these "archetypes" (types of
problems), then matched to a specific recommended action - rather than
treating every flagged page the same way.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess
import pandas as pd

REPO_URL = "https://github.com/darksider747/flyrank-1st"
REPO_DIR = "/content/flyrank-1st"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")
df["ctr_below_tier"] = df["ctr"] < (0.7 * tier_avg_ctr)
df["is_stale"] = df["days_since_last_update"] >= 91

def get_reason_code(row):
    if row["ctr_below_tier"] and row["is_stale"]:
        return "ctr_and_stale"
    elif row["ctr_below_tier"]:
        return "ctr_below_tier_average"
    elif row["is_stale"]:
        return "stale_content"
    else:
        return "no_flag"

df["reason_code"] = df.apply(get_reason_code, axis=1)

action_map = {
    "ctr_and_stale": "high_priority_review",
    "ctr_below_tier_average": "review_title_meta",
    "stale_content": "review_content_refresh",
    "no_flag": "monitor"
}
df["action"] = df["reason_code"].map(action_map)

print(df["reason_code"].value_counts())

Loaded 30000 rows
reason_code
ctr_below_tier_average    15712
ctr_and_stale              7016
no_flag                    4943
stale_content              2329
Name: count, dtype: int64


## 2. Intended use and limits

Intended use: This model is meant for a content reviewer or manager who
doesn't have time to check every single page on a website by hand. It
produces a ranked list, suggesting which pages to look at first - it
does not decide anything on its own, and it does not take any action
by itself (no automatic rewriting, no automatic publishing).

The output must always be treated as a starting point, not a final
answer. A human still needs to look at each flagged page before acting
on it, because of several real issues found across this project:

- Missing tracking data: some pages show zero GA4 sessions simply
  because tracking wasn't set up for that client yet, not because the
  page is actually performing badly (confirmed in Week 3 - only about
  6.3% of rows had GA4 tracking available).

- Low-traffic noise: pages with very few views (under 300) can show a
  misleadingly bad CTR just by chance - 8 of my own top-20 flagged
  pages fell into this category in Week 4.

- Confident but wrong flags: my Week 5 model was very confident (80%+
  sure) about 555 pages that turned out not to actually be declining -
  these tended to have low CTR but otherwise healthy signals, like
  decent position and freshness.

- A leakage-related score inflation: my Week 5 model's reported
  performance (Precision@50 = 0.780) turned out to be partly inflated
  by a feature that likely overlapped with the label - the more honest
  number, after fixing this in Week 6, was 0.560.

Limits:

Primary limit: the model's features were checked for timing overlap
with the label in Week 6 - impressions_90d was removed after
discovering it likely overlapped with the label's own calculation
window, dropping Precision@50 from 0.780 to a more honest 0.560. Any
future retraining should re-check this before trusting reported
performance numbers.

Secondary limits:
- Small sample sizes can produce misleading percentages in thin buckets
  (e.g. Week 4's 181+ freshness tier had only 174 pages).
- Some fields, like days_since_last_update, showed suspicious repeated
  values (many top-ranked pages sharing an identical "104 days" value),
  suggesting a possible data quality issue rather than a true signal.
- Missing tracking data: some pages show zero GA4 sessions simply
  because tracking wasn't set up for that client yet, not because the
  page is actually performing badly (confirmed in Week 3, using the
  full warehouse dataset - only about 6.3% of rows had GA4 tracking
  available; this specific check isn't repeatable on the smaller
  starter CSV used in this notebook, since it doesn't include that
  column).

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ga4_check = df["ga4_data_available"].value_counts() if "ga4_data_available" in df.columns else "column not in starter CSV"
print(ga4_check)

column not in starter CSV


Human review rules:

General rule: Every single page flagged by this model requires human
review before any action is taken. The model only suggests and ranks -
it never judges or decides on its own.

Extra-caution cases (require closer review than usual):

1. Missing GA4 data: if a flagged page has zero or missing session data,
   a reviewer should first check whether GA4 tracking was even active
   for that client during that period, before assuming the page is
   genuinely underperforming.

2. Scores relying heavily on leakage-prone signals: since this model's
   reported performance was found to be partly inflated by a feature
   (impressions_90d) that likely overlapped with the label (Week 6 -
   Precision@50 dropped from 0.780 to 0.560 once removed), any high
   score should be treated with added skepticism until the underlying
   features are re-verified as leakage-free.

No-go list - this model should NEVER be used to:

1. Automatically publish, delete, or edit a page without human review -
   the model only suggests, it does not verify or guarantee correctness.

2. Claim that refreshing a flagged page WILL cause it to recover - this
   would require an actual controlled experiment, which this data
   cannot provide.

3. Treat a single flagged page as a certain, confirmed problem -
   especially low-traffic pages (under ~300 views), where the
   underlying numbers can be noise rather than a real pattern. This is
   not a rare edge case: 9,836 of the 25,057 flagged pages in this
   dataset (about 39%) fall under this low-traffic threshold, so a
   meaningful share of the ranked queue needs this extra scrutiny.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
low_traffic_flagged = df[(df["reason_code"] != "no_flag") & (df["impressions_90d"] < 300)]
print(f"Flagged pages with under 300 impressions_90d: {len(low_traffic_flagged)} out of {len(df[df['reason_code'] != 'no_flag'])} total flagged pages")

Flagged pages with under 300 impressions_90d: 9836 out of 25057 total flagged pages


## 4. Monitoring / retrain triggers

Monitoring trigger: periodically re-calculate Precision@50 on fresh
data (e.g. every month or quarter) and compare it against the current
baseline (0.560, from Week 6's leakage-audited result).

Retrain trigger: if Precision@50 drops meaningfully below this baseline
(for example, by more than 10-15%), that's a signal the model may be
going stale - possibly because content patterns, search behavior, or
client mix have changed since training - and it should be retrained on
more recent data.

Second monitoring trigger - repeat leakage audit: any time the model is
retrained with new or additional features, repeat the same leakage
check performed in Week 6 (comparing Precision@50 with vs without each
new feature). If adding a feature causes a suspiciously large jump in
performance, investigate for timing overlap or other leakage before
trusting the new score.

Third, simpler check - data availability drift: periodically recheck
what fraction of rows have real GA4 tracking (ga4_data_available IS
TRUE), since Week 3 found only ~6.3% of rows had this on the warehouse
data. If this fraction changes significantly over time, features
relying on GA4 data may become more or less reliable than originally
assumed.

These three checks aren't exhaustive, but they directly address the
three biggest real problems found during this project (score staleness,
leakage, and data coverage gaps). A full production system would likely
need more automated monitoring than this notebook-based, non-production
approach.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
current_precision_at_50 = 0.560  # from Week 6's leakage-audited result

print(f"Current baseline Precision@50 (to monitor against): {current_precision_at_50}")
print(f"Retrain trigger threshold (10-15% drop): below {current_precision_at_50 * 0.85:.3f} to {current_precision_at_50 * 0.90:.3f}")

Current baseline Precision@50 (to monitor against): 0.56
Retrain trigger threshold (10-15% drop): below 0.476 to 0.504


## 5. Exports for the paper

Exports:

work/outputs/action_playbook_queue.csv - the full ranked queue (30,000
rows), sorted by reason code priority, with action labels attached.
This file stays out of git (regenerated each run) and is meant to be
regenerated fresh whenever this notebook is re-run.

work/outputs/playbook_metrics.json - the key numbers this playbook's
claims are built on: baseline vs model Precision@50 (before and after
the Week 6 leakage fix), total flagged pages, low-traffic flagged pages,
GA4 tracking coverage, and the computed retrain threshold. This file IS
committed to git, since it serves as the "receipts" my research paper
will quote numbers from next week.

Note: no figures were generated in this notebook since the playbook is
primarily reasoning and reason-code logic rather than visual analysis -
prior weeks' charts (e.g. CTR-by-position, Precision@K comparisons)
remain available in earlier notebooks if needed for the paper.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs("work/outputs", exist_ok=True)

ranked = df.sort_values("reason_code", key=lambda x: x.map({"ctr_and_stale": 2, "ctr_below_tier_average": 1, "stale_content": 1, "no_flag": 0}), ascending=False)
ranked[["content_id", "reason_code", "action", "ctr", "position_tier", "days_since_last_update", "impressions_90d"]].to_csv(
    "work/outputs/action_playbook_queue.csv", index=False
)
print(f"Saved {len(ranked)} ranked rows to work/outputs/action_playbook_queue.csv")
import json

metrics = {
    "baseline_precision_at_50": 0.500,
    "model_precision_at_50_before_leakage_fix": 0.780,
    "model_precision_at_50_after_leakage_fix": 0.560,
    "flagged_pages_total": int((df["reason_code"] != "no_flag").sum()),
    "flagged_pages_low_traffic_under_300": int(low_traffic_flagged.shape[0]) if 'low_traffic_flagged' in dir() else 9836,
    "ga4_tracking_available_pct_warehouse": 6.3,
    "retrain_threshold_precision_at_50": 0.504
}

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics:")
print(json.dumps(metrics, indent=2))

Saved 30000 ranked rows to work/outputs/action_playbook_queue.csv
Saved metrics:
{
  "baseline_precision_at_50": 0.5,
  "model_precision_at_50_before_leakage_fix": 0.78,
  "model_precision_at_50_after_leakage_fix": 0.56,
  "flagged_pages_total": 25057,
  "flagged_pages_low_traffic_under_300": 9836,
  "ga4_tracking_available_pct_warehouse": 6.3,
  "retrain_threshold_precision_at_50": 0.504
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.